# `main.ipynb`: main simulation notebook
Load packages

In [1]:
import jax
from jax import numpy as jnp
from jax import random
import os
from pathlib import Path
from tqdm import tqdm

from utils.criteria import *
from utils.energies import *
from utils.mcmc import *
from utils.modelIO import *
from utils.overlaps import *
from utils.stability import *

Load data and initialize

In [10]:
GRN_PATH = Path('data/eml/eml_fate_network.npy')
GIDS_PATH = Path('data/eml/eml_gene_ids.csv')
SAVE_PATH = Path('results/eml/')
RGRNS_PATH = SAVE_PATH.joinpath('random_networks.npy')
SEED = 1234
bio_grn, gids, N = load_grn(GRN_PATH, GIDS_PATH)
criterion = MetropolisCriterion()

temperatures = jnp.linspace(0.0, 1.0, 21)
zero_field = jnp.zeros(N)

key = random.key(SEED)
keys_4 = random.split(key, num=4) 
rgrn_keys = random.split(keys_4[0], num=100000)
keys_1e2 = random.split(keys_4[1], num=100)
keys_1e3 = random.split(keys_4[2], num=1000)
keys_1e5 = random.split(keys_4[3], num=100000)

Get random networks

In [ ]:
def rand_pair_sim_rgrn(key):
    return randomize_grn(key, bio_grn) # get random network
rand_pair_sim_rgrn_optimized = jax.jit(jax.vmap(rand_pair_sim_rgrn))

rgrns = rand_pair_sim_rgrn_optimized(rgrn_keys)
jnp.save(RGRNS_PATH, rgrns)

Prime functions

In [4]:
# prime function for biological network
def bio_sim_prime(key, T, field, N_steps):
    grn_state = random.randint(key, (), 0, 2**N) # initialize replica state
    _, key = random.split(key) # refresh key
    return get_trajectory(key, grn_state, bio_grn, field, T, criterion, N, N_steps)

# prime function for random networks
def rand_pair_sim_prime(key_rgrn_pair, T, field, N_steps):
    key, rgrn = key_rgrn_pair # unpack key and random network
    grn_state = random.randint(key, (), 0, 2**N) # initialize replica state
    _, key = random.split(key) # refresh key
    return get_trajectory(key, grn_state, rgrn, field, T, criterion, N, N_steps)

Biological GRN: 1000 replicas x 21 temperatures ($N^4$ timesteps each, $h=0$)

In [4]:
bio_vary_T_path = SAVE_PATH.joinpath('bio_vary_T')
bio_vary_T_path.mkdir(parents=True, exist_ok=True)

for T in tqdm(temperatures): # for loop over temperatures due to memory limit
    xs_filename = bio_vary_T_path.joinpath('bio_T_{:.0e}_xs.npy'.format(T))
    vs_filename = bio_vary_T_path.joinpath('bio_T_{:.0e}_vs.npy'.format(T))

    def bio_sim_T_vary(key): # fix h=0, N_steps=N**4
        return bio_sim_prime(key, T, zero_field, N**4)
    bio_sim_T_vary_optimized = jax.jit(jax.vmap(bio_sim_T_vary))

    xs, vs = bio_sim_T_vary_optimized(keys_1e3)
    jnp.save(xs_filename, xs)
    jnp.save(vs_filename, vs)

100%|██████████| 21/21 [1:01:49<00:00, 176.66s/it]


Random GRNs: 1000 GRN-replica pairs x 21 temperatures ($N^4$ timesteps each, $h=0$)

In [33]:
rand_vary_T_path = SAVE_PATH.joinpath('rand_vary_T')
rand_vary_T_path.mkdir(parents=True, exist_ok=True)
rgrns_1e3 = jnp.load(rgrns_filename)[:1000]

for T in tqdm(temperatures): # for loop over temperatures due to memory limit
    xs_filename = rand_vary_T_path.joinpath('rand_T_{:.0e}_xs.npy'.format(T))
    vs_filename = rand_vary_T_path.joinpath('rand_T_{:.0e}_vs.npy'.format(T))

    def rand_pair_sim_T_vary(rand_pair): # fix h=0, N_steps=N**4
        return rand_pair_sim_prime(rand_pair, T, zero_field, N**4)
    rand_sim_T_vary_optimized = jax.jit(jax.vmap(rand_pair_sim_T_vary))

    xs, vs = rand_sim_T_vary_optimized((keys_1e3, rgrns_1e3))
    jnp.save(xs_filename, xs)
    jnp.save(vs_filename, vs)

100%|██████████| 21/21 [1:03:40<00:00, 181.93s/it]


Biological GRN: 100000 replicas ($N^2$ timesteps each, $h=0$)

In [ ]:
tsl_save_path = SAVE_PATH.joinpath('tsl')
tsl_save_path.mkdir(parents=True, exist_ok=True)

bio_many_short_xs_filename = tsl_save_path.joinpath('bio_many_short_xs.npy')
bio_many_short_vs_filename = tsl_save_path.joinpath('bio_many_short_vs.npy')

def bio_many_short(key): # fix T=0, h=0, N_steps=N**2
    return bio_sim_prime(key, 0.0, zero_field, N**2)
bio_many_short_optimized = jax.jit(jax.vmap(bio_many_short))

xs, vs = bio_many_short_optimized(keys_1e5)
jnp.save(bio_many_short_xs_filename, xs)
jnp.save(bio_many_short_vs_filename, vs)

Random GRN: 100000 replicas ($N^2$ timesteps each, $h=0$)

In [11]:
rgrns = jnp.load(RGRNS_PATH)

rand_many_short_xs_filename = tsl_save_path.joinpath('rand_many_short_xs.npy')
rand_many_short_vs_filename = tsl_save_path.joinpath('rand_many_short_vs.npy')

def rand_many_short(rand_pair): # fix T=0, h=0, N_steps=N**2
    return rand_pair_sim_prime(rand_pair, 0.0, zero_field, N**2)
rand_many_short_optimized = jax.jit(jax.vmap(rand_many_short))

xs, vs = rand_many_short_optimized((keys_1e5, rgrns))
jnp.save(rand_many_short_xs_filename, xs)
jnp.save(rand_many_short_vs_filename, vs)